# 🧠 Kairos Cognitive Diffusion: CIDA + DiffusionGemma

Добро пожаловать в практический ноутбук по интеграции плагина **CIDA (Cognitive Interactive Deliberation Agents)** с новейшей дискретной диффузионной языковой моделью от Google DeepMind — **DiffusionGemma**.

### Цель этого ноутбука:
1. Загрузить официальную предобученную модель `google/diffusiongemma-26B-A4B-it` с Hugging Face с помощью класса `DiffusionGemmaForBlockDiffusion`.
2. Применить 4-битное квантование (`bitsandbytes`), чтобы модель поместилась в VRAM стандартной Colab GPU (L4 или T4).
3. Настроить когнитивную обертку `CognitiveDiffusionGemma`, которая внедряет плагин CIDA прямо в процесс генерации (denoising).
4. Продемонстрировать работу многоагентных дебатов для повышения связности параллельной генерации текста.

---
### ⚠️ ВАЖНОЕ ПРИМЕЧАНИЕ:
Модель `DiffusionGemma` является закрытой (gated). Перед запуском:
1. Войдите в свой аккаунт Hugging Face.
2. Перейдите на страницу [google/diffusiongemma-26B-A4B-it](https://huggingface.co/google/diffusiongemma-26B-A4B-it) и подтвердите согласие с лицензионным соглашением.
3. Создайте HF Access Token в настройках аккаунта.

## 1. Подготовка окружения и клонирование репозитория

In [ ]:
# Клонируем репозиторий с исходным кодом CIDA
!git clone https://github.com/Kairatzh/CIDA-plugin.git
%cd CIDA-plugin

In [ ]:
# Устанавливаем зависимости
!pip install -q transformers accelerate bitsandbytes huggingface_hub

## 2. Авторизация в Hugging Face

In [ ]:
# Введите ваш токен от Hugging Face для загрузки весов
from huggingface_hub import notebook_login
notebook_login()

## 3. Загрузка DiffusionGemma (4-bit)

В отличие от стандартных авторегрессионных моделей, DiffusionGemma использует специальный класс `DiffusionGemmaForBlockDiffusion` для работы с блочным диффузионным шумоподавлением.

In [ ]:
import torch
from transformers import DiffusionGemmaForBlockDiffusion, AutoProcessor, BitsAndBytesConfig

model_id = "google/diffusiongemma-26B-A4B-it"

# Настраиваем 4-битное квантование для экономии VRAM
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

print("Загружаем процессор...")
processor = AutoProcessor.from_pretrained(model_id)
tokenizer = processor.tokenizer

print("Загружаем модель DiffusionGemma (это может занять несколько минут)...")
base_model = DiffusionGemmaForBlockDiffusion.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto"
)
print("Модель успешно загружена!")

## 4. Обертка CognitiveDiffusionGemma

Мы создаем класс, который перехватывает скрытые состояния последнего слоя Gemma MoE, передает их агентам CIDA для дебатов по всей последовательности, а затем проецирует консенсус обратно в оригинальный `lm_head` модели Gemma.

In [ ]:
import torch.nn as nn
from cida_plugin.core import CIDAPlugin
from cida_plugin.config import CIDAPluginConfig

class CognitiveDiffusionGemma(nn.Module):
    def __init__(self, base_model, d_hidden=256):
        super().__init__()
        self.base_model = base_model
        self.d_model = base_model.config.hidden_size
        
        # Инициализируем CIDA
        # Мы используем dummy num_classes=2, так как классификацию токенов 
        # делает оригинальный lm_head модели Gemma, а CIDA оптимизирует скрытые состояния
        self.cida_config = CIDAPluginConfig(
            d_input=self.d_model,
            d_hidden=d_hidden,
            num_classes=2,
            multi_label=False,
            max_rounds=2
        )
        self.cida = CIDAPlugin(self.cida_config)
        
        # Проекция из консенсуса CIDA обратно в пространство Gemma d_model
        self.out_proj = nn.Linear(d_hidden, self.d_model)
        
    def forward(self, input_ids, decoder_input_ids=None, attention_mask=None, decoder_attention_mask=None):
        # 1. Прогоняем данные через оригинальные MoE слои
        with torch.no_grad():
            outputs = self.base_model(
                input_ids=input_ids,
                decoder_input_ids=decoder_input_ids,
                attention_mask=attention_mask,
                decoder_attention_mask=decoder_attention_mask,
                output_hidden_states=True,
                return_dict=True
            )
            
        # Скрытые состояния последнего слоя (поддержка разных имен выходных тензоров)
        H = None
        if hasattr(outputs, "decoder_hidden_states") and outputs.decoder_hidden_states is not None:
            H = outputs.decoder_hidden_states[-1]
        elif hasattr(outputs, "hidden_states") and outputs.hidden_states is not None:
            H = outputs.hidden_states[-1]
        elif hasattr(outputs, "encoder_hidden_states") and outputs.encoder_hidden_states is not None:
            H = outputs.encoder_hidden_states[-1]
            
        if H is None:
            raise ValueError("Не удалось извлечь скрытые состояния из выходов модели.")
            
        B, L, d_model = H.shape
        
        # 2. Готовим данные для CIDA
        # Чтобы каждый токен общался со всей последовательностью, 
        # мы дублируем контекст для каждого токена
        H_seq = H.unsqueeze(1).expand(B, L, L, self.d_model).reshape(B * L, L, self.d_model)
        H_flat = H.reshape(B * L, self.d_model)
        
        # 3. Запускаем когнитивные дебаты
        cida_out = self.cida(pooled=H_flat, seq_output=H_seq)
        
        # Извлекаем финальные состояния агентов
        # state_final.s shape: (B * L, M, d_hidden)
        s_final = cida_out["state_final"].s.mean(dim=1) # (B * L, d_hidden)
        
        # 4. Проецируем обратно в пространство Gemma и восстанавливаем размерность последовательности
        H_refined = self.out_proj(s_final).view(B, L, self.d_model)
        
        # 5. Классифицируем через оригинальный lm_head
        logits = self.base_model.lm_head(H_refined) # (B, L, vocab_size)
        
        return logits, cida_out["disagreement"].view(B, L)

print("Когнитивная обертка успешно определена!")

## 5. Запуск Теста

Мы создаем экземпляр когнитивной диффузии и делаем пробный шаг генерации на случайном промпте.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cognitive_model = CognitiveDiffusionGemma(base_model).to(device)

prompt = "The key to artificial general intelligence is"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

# Для тестирования encoder-decoder структуры DiffusionGemma, 
# создаем временную маску (decoder_input_ids) длиной 8 токенов
mask_token_id = getattr(tokenizer, "mask_token_id", 103)
decoder_input_ids = torch.full((1, 8), mask_token_id, dtype=torch.long, device=device)

# Выполняем когнитивный forward pass
logits, disagreement = cognitive_model(
    input_ids=inputs["input_ids"],
    decoder_input_ids=decoder_input_ids,
    attention_mask=inputs.get("attention_mask")
)

print(f"Входные токены (Prompt): {inputs['input_ids'].shape}")
print(f"Входные маски (Canvas): {decoder_input_ids.shape}")
print(f"Выходные логиты: {logits.shape}")
print(f"Среднее несогласие агентов CIDA на токен: {disagreement.mean().item():.4f}")

## 6. Визуализация разнообразия агентов

Давайте проверим, насколько мнения агентов расходятся в зависимости от токенов промпта. Высокое несогласие обычно сигнализирует о сложной семантической развилке.

In [ ]:
# Токены холста дешумирования
canvas_tokens = [f"[MASK_{i}]" for i in range(decoder_input_ids.shape[1])]
dis_values = disagreement[0].detach().cpu().numpy()

print("Анализ несогласия агентов на холсте дешумирования:")
print("-" * 50)
for tok, val in zip(canvas_tokens, dis_values):
    print(f"Позиция холста: {tok:<15} | Несогласие (Disagreement): {val:.4f}")